# nb_bronze_stream_vehicle_battery (Kafka connector)

**Source :** Azure Event Hub — `vehicle-battery-live` via Kafka endpoint  
**Sink   :** `abfss://bronze@evdatalakedev2.dfs.core.windows.net/event-stream/vehicle_battery_live/`  
**Format :** Delta (append), partitioned by `event_date`  
**Trigger:** Every 30 seconds  

## Why Kafka connector?

The native EventHubs connector (`azure-eventhubs-spark`) only supports Scala 2.12.  
DBR 18 / Spark 4.1 uses **Scala 2.13** — the JAR is incompatible.  
The built-in Kafka connector works on **all DBR versions** including Scala 2.13.  
No Maven library install required.

**Key Vault secret needed:** `eventhub-vehicle-battery-listen-conn-str`  
Value: full SAS connection string for `databricks-listen-policy` (with EntityPath)

| Kafka column | Description |
|---|---|
| `value` | Raw JSON bytes (event payload) |
| `timestamp` | Event Hub enqueue time |
| `partition` | Partition number |
| `offset` | Byte offset within partition |
| `topic` | Event Hub topic name |

In [ ]:
# ── Cell 1: secrets ───────────────────────────────────────────────────────────
EH_CONN_STR  = dbutils.secrets.get(scope="ev-kv-scope", key="eventhub-vehicle-battery-listen-conn-str")

SP_CLIENT_ID  = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-id")
SP_CLIENT_SEC = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-secret")
SP_TENANT_ID  = dbutils.secrets.get(scope="ev-kv-scope", key="sp-tenant-id")
ADLS_ACCOUNT  = dbutils.secrets.get(scope="ev-kv-scope", key="adls-account-name")

print("Secrets loaded.")

In [ ]:
# ── Cell 2: ADLS OAuth config ─────────────────────────────────────────────────
spark.conf.set(f"fs.azure.account.auth.type.{ADLS_ACCOUNT}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{ADLS_ACCOUNT}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{ADLS_ACCOUNT}.dfs.core.windows.net",     SP_CLIENT_ID)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{ADLS_ACCOUNT}.dfs.core.windows.net", SP_CLIENT_SEC)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{ADLS_ACCOUNT}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token")
print("ADLS OAuth configured.")

In [ ]:
# ── Cell 3: paths + Kafka/Event Hub config ────────────────────────────────────
EH_NAMESPACE = "evh-ev-intelligence-dev"
EH_TOPIC     = "vehicle-battery-live"
EH_BOOTSTRAP = f"{EH_NAMESPACE}.servicebus.windows.net:9093"

conn_str = EH_CONN_STR.strip()

# Databricks uses a shaded Kafka — must use kafkashaded prefix for the LoginModule
SASL_CONFIG = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    'username="$ConnectionString" '
    f'password="{conn_str}";'
)

BRONZE_PATH     = f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/event-stream/vehicle_battery_live/"
CHECKPOINT_PATH = f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/_checkpoints/vehicle-battery-live/"

KAFKA_OPTIONS = {
    "kafka.bootstrap.servers":      EH_BOOTSTRAP,
    "subscribe":                    EH_TOPIC,
    "kafka.security.protocol":      "SASL_SSL",
    "kafka.sasl.mechanism":         "PLAIN",
    "kafka.sasl.jaas.config":       SASL_CONFIG,
    "startingOffsets":              "latest",
    "maxOffsetsPerTrigger":         1000,
    "failOnDataLoss":               "false",
    "kafka.request.timeout.ms":     "60000",
    "kafka.session.timeout.ms":     "60000",
}

print(f"Bootstrap : {EH_BOOTSTRAP}")
print(f"Topic     : {EH_TOPIC}")
print(f"Bronze    : {BRONZE_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

In [ ]:
# ── Cell 4: read stream via Kafka connector ───────────────────────────────────
raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**KAFKA_OPTIONS)
    .load()
)

print("Raw Kafka stream schema:")
raw_stream.printSchema()

In [ ]:
# ── Cell 5: parse JSON from the Kafka value column ────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

EVENT_SCHEMA = StructType([
    StructField("event_id",                   StringType(),  True),
    StructField("vehicle_id",                 StringType(),  True),
    StructField("session_id",                 StringType(),  True),
    StructField("station_id",                 StringType(),  True),
    StructField("charger_id",                 StringType(),  True),
    StructField("battery_pct",                DoubleType(),  True),
    StructField("charging_rate_kw",           DoubleType(),  True),
    StructField("battery_temp_c",             DoubleType(),  True),
    StructField("state_of_charge_target_pct", IntegerType(), True),
    StructField("estimated_minutes_to_full",  IntegerType(), True),
    StructField("event_ts",                   StringType(),  True),
])

parsed = (
    raw_stream
    .withColumn("value_str", F.col("value").cast("string"))
    .withColumn("parsed",    F.from_json(F.col("value_str"), EVENT_SCHEMA))
    .select(
        F.col("parsed.*"),
        F.col("timestamp").alias("eh_enqueued_time"),
        F.col("partition").alias("eh_partition"),
        F.col("offset")   .alias("eh_offset"),
        F.col("topic")    .alias("eh_topic"),
    )
)

print("Parsed schema:")
parsed.printSchema()

In [ ]:
# ── Cell 6: add Bronze metadata columns ───────────────────────────────────────
bronze_df = (
    parsed
    .withColumn("_ingestion_ts", F.current_timestamp())
    .withColumn("_source",       F.lit("eventhub/vehicle-battery-live"))
    .withColumn("_is_corrupt",   F.col("event_id").isNull())
    .withColumn("event_date",    F.to_date(F.col("event_ts")))
)

print("Bronze schema:")
bronze_df.printSchema()

In [ ]:
# ── Cell 7: write stream to Bronze Delta ──────────────────────────────────────
streaming_query = (
    bronze_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .partitionBy("event_date")
    .trigger(processingTime="30 seconds")
    .start(BRONZE_PATH)
)

print(f"Stream started. Query ID: {streaming_query.id}")

In [ ]:
# ── Cell 7a: clear checkpoint if stream keeps failing (run once, then delete) ──
# Only run this if the stream refuses to start after fixing config
# It forces a clean start — will re-read from "latest" offset
dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)
print(f"Checkpoint cleared: {CHECKPOINT_PATH}")

In [ ]:
# ── Cell 8: monitor loop ───────────────────────────────────────────────────────
import time

while streaming_query.isActive:
    prog = streaming_query.lastProgress
    if prog:
        print(f"[Batch {prog.get('batchId','?')}] inputRows={prog.get('numInputRows',0)}  rate={prog.get('inputRowsPerSecond',0.0):.1f} rows/s")
    time.sleep(30)

streaming_query.awaitTermination()

In [ ]:
# ── Cell 9: verify (run manually) ─────────────────────────────────────────────
verify_df = spark.read.format("delta").load(BRONZE_PATH)
print(f"Total Bronze rows: {verify_df.count()}")
verify_df.orderBy(F.col("_ingestion_ts").desc()).limit(5).display()